# 09 — Advanced Indexing (Multi-Vector Summaries, Parent-Child & Persistence)

**By the end of this notebook, you can decouple small search vectors from large full-context parent blocks and persist vector indexes cleanly.**

### Learning Objectives
- Solve the Small-vs-Large chunk dilemma using Parent-Child document retrieval.
- Build a Multi-Vector Summary Index pointing distilled summaries to complete source files.
- Persist and reload complete vector stores safely without external database crashes.

### Prerequisites
- Notebooks 00–08

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Search Small Chunks, Deliver Full Context

Earlier we hit a tension: **small chunks** (120 chars) match search queries cleanly, but lose surrounding explanations; **large chunks** (800 chars) give rich LLM context, but dilute embeddings!

**Parent-Child & Multi-Vector Indexing** gives you both worlds using a **book index analogy**:
- You search the crisp **1-line index entry (Child / Summary Vector)**.
- Then you look up the **doc_id page number** and hand the **entire 600-char section (Parent Document)** to the LLM!

```
Tiny Child Chunk (120 chars)  ---> Embedded in Vector Store (High Search Precision!)
       |
       +--- carries metadata["parent_id"] ---> In-Memory DocStore [Full Parent Block (600 chars)]
                                                                         |
                                                                         v
                                                              Handed to LLM for rich answers!
```

## 3. Step-by-Step Build

### Step 1: Build Parent Blocks and Small Linked Child Chunks
Let's slice our corpus into larger parent blocks (650 chars), store them by ID in a `parent_store` dictionary, and chop each parent into crisp 140-char child chunks.

In [2]:
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

parents = RecursiveCharacterTextSplitter(chunk_size=650, chunk_overlap=60).split_documents(load_corpus_documents(data_dir))
parent_store = {}
child_chunks = []
child_splitter = RecursiveCharacterTextSplitter(chunk_size=140, chunk_overlap=20)
for p in parents:
    pid = str(uuid.uuid4())[:8]
    parent_store[pid] = p
    for c in child_splitter.split_documents([p]):
        c.metadata["parent_id"] = pid
        child_chunks.append(c)
print(f"Created {len(parent_store)} parents and {len(child_chunks)} precision child chunks.")
        

Created 16 parents and 84 precision child chunks.


> **What just happened?**
> Each tiny child chunk knows its exact parent via `metadata['parent_id']`.

### Step 2: Retrieve via Child Vectors, Hydrate Parent Blocks
Watch how searching small child chunks pinpoints a tiny phrase (`40 milliseconds`), then returns the complete surrounding parent paragraph cleanly.

In [3]:
from langchain_core.vectorstores import InMemoryVectorStore
child_index = InMemoryVectorStore.from_documents(child_chunks, embeddings)

def parent_retrieve(query: str, k: int = 2):
    seen, results = set(), []
    for ch in child_index.similarity_search(query, k=k * 2):
        pid = ch.metadata["parent_id"]
        if pid not in seen:
            seen.add(pid); results.append(parent_store[pid])
        if len(results) >= k: break
    return results

parents_found = parent_retrieve("ZeroGravity failover threshold", k=2)
print("Child match ~140 chars -> Hydrated Parent chars:", len(parents_found[0].page_content))
        

Child match ~140 chars -> Hydrated Parent chars: 437


> **What just happened?**
> Deduplicating `parent_id` guarantees we never return two copies of the same parent even if two children matched!

### Step 3: Multi-Vector Summary Index
Instead of indexing raw fragments, we can ask the LLM to write clean document summaries, embed the summaries, and link each back to the full file.

In [4]:
from langchain_core.documents import Document
full_docs = load_corpus_documents(data_dir)
summary_docs = []
for i, d in enumerate(full_docs):
    summ = llm.invoke(f"Write a concise summary of:\nDOCUMENT:\n{d.page_content[:900]}").content
    summary_docs.append(Document(page_content=summ, metadata={"doc_idx": i, "source": d.metadata["source"]}))

summary_index = InMemoryVectorStore.from_documents(summary_docs, embeddings)
top_summary = summary_index.similarity_search("ROS2 distributions and failover", k=1)[0]
print("Summary index routed to full doc:", top_summary.metadata["source"])
        

Summary index routed to full doc: product_faq.md


> **What just happened?**
> Multi-Vector indexing shines on noisy tables, code listings, or messy PDFs where concise summaries embed far more reliably.

### Step 4: Persist & Reload Vector Index Cleanly
Let's serialize `child_index` cleanly to JSON on disk and verify identical retrieval immediately after reload.

In [5]:
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    save_file = str(Path(tmp) / "child_index.json")
    child_index.dump(save_file)
    restored = InMemoryVectorStore.load(save_file, embeddings)
q = "How much is the annual learning stipend?"
assert restored.similarity_search(q, k=1)[0].page_content == child_index.similarity_search(q, k=1)[0].page_content
print("JSON index persistence verified cleanly with exact parity!")
        

JSON index persistence verified cleanly with exact parity!


> **What just happened?**
> JSON persistence avoids binary C++/Rust database locks on Windows while staying fast for course & prototyping scale.

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Compare raw child length vs parent length** — *Hint:* Print the first 80 chars of the top matched child chunk alongside its full parent.
2. **Inspect summary index accuracy** — *Hint:* Query `summary_index` with `physical safety perimeter E-Stop`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
top_s = summary_index.similarity_search("physical safety perimeter E-Stop", k=1)[0]
print("Summary hit:", top_s.metadata["source"], "->", top_s.page_content[:100])
        

Summary hit: policy_brief.pdf -> Summary: Novasphere Dynamics - Autonomous Robotics Safety & Deployment Brief Document ID: SAF-2026-B


## 5. Common Mistakes
- Returning duplicate parent blocks when 3 child chunks all belong to the same parent — always deduplicate by `parent_id`.
- Forgetting to save `parent_store` alongside `child_index` when deploying persisted multi-vector retrieval.
- Summarizing huge documents naively inside tight rate limits without batching.

## 6. Recap
- Parent-Child indexing gives high retrieval accuracy (small chunks) AND complete context (large parents).
- Multi-Vector summary indexing separates searchable representations from raw storage.
- `InMemoryVectorStore.dump()` and `.load()` provide zero-crash pure-Python persistence.

## 7. Next Notebook
Next up (`10_reranking_and_compression.ipynb`): Filter out near-miss chunks using Two-Stage Reranking and strip irrelevant filler sentences via Contextual Compression!

## 8. Video Chapter Outline
1. **The Small-vs-Large Chunk Paradox** — Why single chunk sizes force compromises.
2. **Parent-Child Implementation** — Linking small chunks to parent UUIDs cleanly.
3. **Multi-Vector Summary Indexing** — Indexing LLM summaries over raw messy documents.
4. **Safe JSON Index Persistence** — Dumping and reloading vector stores cleanly.
5. **Wrap-up** — Production storage best practices.